In [ ]:
import Pkg;
Pkg.activate("../");

In [ ]:
using RigorousInvariantMeasures

In [ ]:
#Pkg.rm("BallArithmetic")

In [ ]:
#Pkg.add(url="https://github.com/JuliaBallArithmetic/BallArithmetic.jl", rev="OtherStrategies")

In [ ]:
using BallArithmetic, Plots

In [ ]:
B = RigorousInvariantMeasures.FourierAnalytic(256, 32768)

In [ ]:
D = mod1_dynamic(x->2*x+(1/2π-1/64)*RigorousInvariantMeasures.sinpi(2*x)+0.25)

In [ ]:
plot(x -> plottable(D,x), 0, 1)
plot!(x -> x)

In [ ]:
P = DiscretizedOperator(B, D)

In [ ]:
#Pkg.add("IntervalArithmetic")

In [ ]:
using IntervalArithmetic
midI = IntervalArithmetic.mid
radI = IntervalArithmetic.radius

In [ ]:
midP = midI.(real.(P.L)) + im * midI.(imag.(P.L))

In [ ]:
radP = sqrt.(radI.(real.(P.L))^2 + radI.(imag.(P.L))^2)

In [ ]:
BallP = BallMatrix(midP, radP)

In [ ]:
using Pseudospectra, Plots

In [ ]:
spectralportrait(midP)

In [ ]:
#midPU = Matrix(midI.(PU.L))

In [ ]:
#spectralportrait(midPU)

In [ ]:
savefig("pseudospectra_dfly.pdf")

In [ ]:
using BallArithmetic

In [ ]:
enc, errF, errT, norm_Z, norm_Z_inv = BallArithmetic.compute_enclosure_circles(BallP, 0.3, 1.1, 0.00001, max_steps=4000, rel_steps=1024, rel_pearl_size=1 / 1024)

In [ ]:
BallArithmetic.bound_resolvent(enc[4], errF, errT, norm_Z, norm_Z_inv)

In [ ]:
function certify_enclosure(enc, discr_error, weak_strong, errF, errT, norm_Z, norm_Z_inv)
    N = length(enc.points)

    r = Inf
    for i in 1:N
        abs_z = abs(Ball(enc.points[i], enc.radiuses[i]))
        r = min(r, BallArithmetic.sub_down(abs_z.c, abs_z.r))
    end
    δ = BallArithmetic.bound_resolvent(enc, errF, errT, norm_Z, norm_Z_inv)
    left_side = BallArithmetic.div_down(r) # @down
    right_side = BallArithmetic.mul_up(BallArithmetic.mul_up(weak_strong, discr_error), δ) #up
    if left_side > right_side
        @info "The enclosure of ", enc.λ, "is certified"
        return true
    else
        return false
    end
end

In [ ]:
discr_error = 1.5e-83
weak_strong = 1657.0
certify_enclosure.(enc[1:9], discr_error, weak_strong, errF, errT, norm_Z, norm_Z_inv)

In [ ]:
#enc = BallArithmetic.compute_enclosure(BallMatrix(Q), 0.4, 1.1, 0.001, max_steps = 2000, rel_steps = 64)

In [ ]:
enc

In [ ]:
enc[1][2]

In [ ]:
using Plots, LinearAlgebra

In [ ]:
pl = plot()

pl = scatter!(pl, eigen(midP).values, color=:red, label="", markersize=2.0)

for j in 1:6

    for i in 1:length(enc[j].points)-1
        center_x = real(enc[j].points[i])
        center_y = imag(enc[j].points[i])


        radius = 5 * abs(enc[j].points[i+1] - enc[j].points[i]) / 8

        plot!(pl, [center_x + radius * cos(t) for t in 0:0.1:2π], [center_y + radius * sin(t) for t in 0:0.1:2π], color=:blue, label="")
    end

end

circle_1 = [cos(θ) for θ in 0:0.001:2π], [sin(θ) for θ in 0:0.001:2π]
plot!(circle_1[1], circle_1[2], label="", color=:green)


display(pl)


In [ ]:
savefig("enclosure_blaschke.pdf")

In [ ]:
function bound_back(ϵ, M, r)
    bepsilon = Ball(ϵ)
    bM = Ball(M)
    br = Ball(r)

    bound1 = (Ball(1.0) + bepsilon)^2 * (Ball(1.0) + Ball(2.0) * bepsilon * br) * bM

    return bound1 / (Ball(1.0) - bepsilon * bound1)
end

In [ ]:
bound_back(10^(-9), 1120, 1.1)